## 10_task_a — Job task: read practice_customers, write practice_customers_clean

Used by: practice-job-1-daily-notebook (task_a), practice-job-2-multitask (a)
Config: CATALOG/SCHEMA/PARTITION/CONFIG_PATH widgets -> setup_config() (C360-style)
PARTITION: all | latest | prev | YYYY-MM-DD


In [ ]:

%run ../lib/config_lib


In [ ]:
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (OLD schema name — unchanged by migration)")
dbutils.widgets.text("PARTITION", "all", "all | latest | prev | YYYY-MM-DD")
dbutils.widgets.text("CONFIG_PATH", "", "Optional env config file (empty = built-in dev/stg/prd)")

CFG = setup_config(
    config_path=dbutils.widgets.get("CONFIG_PATH").strip(),
    run_date=dbutils.widgets.get("PARTITION").strip(),
    schema=dbutils.widgets.get("SCHEMA").strip(),
)


In [ ]:

from pyspark.sql import functions as F

src = CFG.table("practice_customers")

df = (
    CFG.partition_filter(spark.read.table(src), table=src)
    .filter(F.col("customer_id").isNotNull())
    .withColumn("segment", F.lower(F.trim(F.col("segment"))))
    .withColumn("monthly_revenue", F.abs(F.col("monthly_revenue")))
    .withColumn("churn_risk", F.least(F.greatest(F.col("churn_risk"), F.lit(0.0)), F.lit(1.0)))
)

CFG.write_partitioned(df, "practice_customers_clean")

dst = CFG.table("practice_customers_clean")
n = spark.read.table(dst).count()
print(f"TASK A DONE — wrote {n} rows to {dst} (PARTITION={CFG.run_date})")
